# Use Case 1 - Name/M2 --> Proposal/Teams

<ol>
    <li>Generic Data Import. <a href="#gen_data_import">Here.</a></li>
    <li>Method/Files Import. <a href="#method_import">Here.</a></li>
    <li>M2 - Mapper Matching. <a href="#m2">Here.</a></li>
</ol>

## Generic Data Import <a id='gen_data_import'></a> 

In [1]:
import pandas as pd
import numpy as np
# Import list of researchers
og_researchers=pd.read_csv('../data/v1_input_files/v1_researchers.csv')

# Import a subset of the proposals, sorted by the YEAR they were sent
proposal_info=pd.read_csv('../data/v1_input_files/v1_proposal_links_title_synopsis.csv')
proposal_info.sort_values(["nsf_proposal_links_v1"], 
                    ascending=[False], 
                    inplace=True)
# proposal_info=proposal_info[:100]
proposal_info.pop("Unnamed: 0")
proposal_info.reset_index(drop=True, inplace=True)

In [2]:
# error case encountered at the end - so preventing it now by excluding this proposal

error_values = ['https://www.nsf.gov/funding/pgm_summ.jsp?pims_id=505073'] 

#drop rows that contain any value in the list
proposal_info = proposal_info[proposal_info.nsf_proposal_links_v1.isin(error_values) == False]
proposal_info.reset_index(drop=True, inplace=True)

## Methods/Files Import <a id='method_import'></a> 

In [3]:
pip install fuzzywuzzy

Note: you may need to restart the kernel to use updated packages.


In [4]:
pip install flask

Note: you may need to restart the kernel to use updated packages.


In [5]:
from mapper4_main.app.main import *
import nlp_techniques
import M2
import importlib
importlib.reload(M2)

[nltk_data] Downloading package punkt to /Users/tej/nltk_data...
[nltk_data]   Package punkt is already up-to-date!
/opt/homebrew/Cellar/jupyterlab/4.4.7/libexec/lib/python3.13/site-packages/fuzzywuzzy/fuzz.py:11: UserWarning: Using slow pure-python SequenceMatcher. Install python-Levenshtein to remove this warning
  warnings.warn('Using slow pure-python SequenceMatcher. Install python-Levenshtein to remove this warning')
[nltk_data] Downloading package wordnet to /Users/tej/nltk_data...


home_dir ./mapper4_main/data/


[nltk_data]   Package wordnet is already up-to-date!
[nltk_data] Downloading package omw-1.4 to /Users/tej/nltk_data...
[nltk_data]   Package omw-1.4 is already up-to-date!


<module 'M2' from '/Users/tej/Desktop/Recommendation_Projects/Teaming/code/M2.py'>

## M2 - Mapper Matching <a id="m1"></a>

<b>Steps:</b>
<ol>
    <li>Extract and preprocess a researcher's own skills (extracted from their homepage in the faculty directory). <a href="#method_import_1">Here.</a></li>
    <li>Store all researchers' skills in a separate list. <a href="#method_import_2">Here.</a></li>
    <li>Extract necessary "skills" required for each proposal, with everyone as a whole having them. <a href="#method_import_3">Here.</a></li>
    <li>Group skills using respective ACM/JEL classification codes. <a href="#method_import_4">Here.</a></li>
    <li>Create teams for each proposal call and for each researcher based on proposal/researcher mapper matches. <a href="#method_import_5">Here.</a></li>
    <li>Apply Ultra-Metric. <a href="#method_import_6">Here.</a></li>
    <li>Final exports to CSV. <a href="#method_import_7">Here.</a></li>    
</ol>

### Step 1 - Extract and preprocess a researcher's own skills (extracted from their homepage in the faculty directory). <a id='method_import_1'></a> 

In [6]:
import ast
import datetime

# Start time
print("Start time:\t", datetime.datetime.now())

m2_researcher_skills={}

# for each researcher
for i in range(len(og_researchers["research"])):
    # load into variables
    researcher=og_researchers["names"][i]
    interests=og_researchers["research"][i]
    if type(interests)==float:  # account for nan values
        interests="['research', 'general', 'computer', 'science', 'engineering']"
    
    # convert the string of interests into a list of interests
    interests=ast.literal_eval(interests)[0].split(", ")
    for j in range(len(interests)):
        interests[j]=nlp_techniques.preprocess(interests[j])
        
    while '' in interests:
        interests.remove('')
    
    # print(researcher,interests)   # E.g., Agostinelli, Forest ['artificial intelligence', 'deep learning', ...]

    # do it for n-grams of 2 as well
    n_gram_interests=[]
    for j in range(len(interests)):
        n_gram_interests.append(nlp_techniques.generate_N_grams(interests[j], ngram=2))
        
    n_gram_interests=[item for sublist in n_gram_interests for item in sublist]   # merge list of lists into a flat list
    
    # merge interests
    merged_interests=set(interests+n_gram_interests)
    while '' in merged_interests:
        merged_interests.remove('')    # remove null/empty strings
    
    # save info
    m2_researcher_skills[researcher]=merged_interests
    
# End time
print("End time:\t", datetime.datetime.now())

Start time:	 2026-04-01 15:28:30.181807
End time:	 2026-04-01 15:28:33.550719


In [7]:
# save directory
save_dir="../data/v1_output_teaming/teaming_1698proposals_316researchers/data_uc1_m2/"

# export m2_researcher_skills
csv_m2_researcher_skills=[]
for i in m2_researcher_skills:
    csv_m2_researcher_skills.append([i, m2_researcher_skills[i]])

csv_m2_researcher_skills=pd.DataFrame(csv_m2_researcher_skills, columns = ['researcher_name', 'skills'])
csv_m2_researcher_skills.to_csv(save_dir+'m2_researcher_skills.csv', encoding='utf-8')
del csv_m2_researcher_skills

### Step 2 - Store all researchers' skills in a separate list. <a id='method_import_2'></a> 

**Why?**

Because a proposal individually would have a lot of "skills" extracted, including the irrelevant terms (or those that the researchers are **not** familiar with). If a proposal has N skills extracted from its title/synopsis, then each of the i-th skill would only count IF that i-th skill is already in m2_all_researcher_skills[] (below).

This way, we are condensing the number of searches, and making Ultra-Metric more readable.

In [8]:
# Start time
print("Start time:\t", datetime.datetime.now())

# set of all skills that researchers have
m2_all_researcher_skills=[]

for i in m2_researcher_skills:    # compile a list of all skills that researchers have
    for j in m2_researcher_skills[i]:
        if j not in m2_all_researcher_skills:
            m2_all_researcher_skills.append(j)

# End time
print("End time:\t", datetime.datetime.now())

Start time:	 2026-04-01 15:28:33.572353
End time:	 2026-04-01 15:28:33.865910


In [9]:
m2_all_researcher_skills

['deep learning',
 'search',
 'bioinformatics',
 'reinforcement learning',
 'artificial intelligence',
 'growth study',
 'electronic technology',
 'position senior',
 'aluminum content',
 'wide bandgap',
 'high aluminum',
 'texas tech',
 'inc ph',
 'content algan',
 'inc ph texas tech universityresearch growth study ultra wide bandgap semiconductor including high aluminum content algan',
 'novel high',
 'tech universityresearch',
 'boron nitride gallium oxide',
 'bandgap semiconductor',
 'high power',
 'electronic photonic',
 'previous position senior scientist',
 'including high',
 'photonic device',
 'fabrication novel high power electronic photonic device',
 'study ultra',
 'ultra wide',
 'fabrication novel',
 'ph texas',
 'computer simulation device',
 'nitride gallium',
 'universityresearch growth',
 'previous position',
 'boron nitride',
 'simulation device',
 'gallium oxide',
 'power electronic',
 'senior scientist',
 'computer simulation',
 'sensor electronic',
 'semiconductor 

In [10]:
# export m2_all_researcher_skills
csv_m2_all_researcher_skills=[]
for i in m2_all_researcher_skills:
    csv_m2_all_researcher_skills.append(i)

csv_m2_proposal_skills=pd.DataFrame(csv_m2_all_researcher_skills, columns = ['all_skills'])
csv_m2_proposal_skills.to_csv(save_dir+'m2_all_researcher_skills.csv', encoding='utf-8')
del csv_m2_all_researcher_skills

### Step 3 - Extract necessary "skills" required for each proposal, with everyone as a whole having them. <a id='method_import_3'></a> 

In [11]:
# Start time
print("Start time:\t", datetime.datetime.now())

m2_proposal_skills={}

# for each proposal
for i in range(len(proposal_info["nsf_proposal_links_v1"])):
    # extract respective title/synopsis
    title=proposal_info["title"][i]
    synopsis=proposal_info["synopsis"][i]
        
    # check if title is an empty field
    if type(title)==float:     # if so, then assign a general value to title
        title="general"
    
    # check if synopsis is an empty field
    if type(synopsis)==float:     # if so, then assign an empty value to synopsis
        synopsis=""
    
    # preprocess title/synopsis
    title=nlp_techniques.preprocess(title)
    synopsis=nlp_techniques.preprocess(synopsis)
        
    # keywords of title - just split the string and apply set()
    keywords=title.split(" ")+synopsis.split(" ")

    # n-gram keywords
    title_n=nlp_techniques.generate_N_grams(title, ngram=2)
    synopsis_n=nlp_techniques.generate_N_grams(synopsis, ngram=2)
    
    n_gram_keywords=set(title+synopsis)
    
    # merge
    all_keywords=set(keywords+title_n+synopsis_n)
    
    # if any of these keywords do not exist in m2_all_researcher_skills, remove them
    skills_to_be_removed=[]
    for j in all_keywords:
        if j not in m2_all_researcher_skills:
            skills_to_be_removed.append(j)
            
    for j in skills_to_be_removed:
        all_keywords.remove(j)        
    
    # in case of empty set
    if all_keywords==set():
        all_keywords=set(["general"])
        
    # add them to the dictionary mapping
    m2_proposal_skills[proposal_info["nsf_proposal_links_v1"][i]]=all_keywords
    
# End time
print("End time:\t", datetime.datetime.now())

Start time:	 2026-04-01 15:28:33.910644
End time:	 2026-04-01 15:28:41.769864


In [12]:
# export m2_proposal_skills
csv_m2_proposal_skills=[]
for i in m2_proposal_skills:
    csv_m2_proposal_skills.append([i, m2_proposal_skills[i]])

csv_m2_proposal_skills=pd.DataFrame(csv_m2_proposal_skills, columns = ['nsf_proposal_links_v1', 'skills'])
csv_m2_proposal_skills.to_csv(save_dir+'m2_proposal_skills.csv', encoding='utf-8')
del csv_m2_proposal_skills

***
### Step 4 - Group skills using respective ACM/JEL classification codes. <a id='method_import_4'></a> 

<ul>
    <li><a href="#method_import_4.1">4.1.</a> For each of the proposal skills, determine their appropriate ACM/JEL classifications.</li>
    <li><a href="#method_import_4.2">4.2.</a> For each of the researchers' skills, determine their appropriate ACM/JEL classifications.</li>
    <li><a href="#method_import_4.3">4.3.</a> Check what categories each of the proposal/researchers' skills belong to.</li>
</ul>

### Step 4.1 - For each of the proposal skills, determine their appropriate ACM/JEL classifications. <a id='method_import_4.1'></a> 

<ul>
    <li>For each of the skills in proposals, identify what category they belong to.</li>
</ul>

In [13]:
# Start time
print("Start time:\t", datetime.datetime.now())

# for each of the skills in proposals, identify what category they belong to.
m2_proposal_skills_mapper={}   # {proposal: [code, term]}

# for each proposal
for proposal in m2_proposal_skills:
    
    m2_proposal_skills_mapper[proposal]={}
    # for each skill
    for skill in m2_proposal_skills[proposal]:
        
        # call mapper
        results=list(M2.callMapper(skill, 0.3, "acm"))    # results = [[codes], [[terms1 for code1], [...], ...] => too complicated 
        
        # make the terms more understandable
        terms=[i for sublist in results[1] for i in sublist]
        while '' in terms:
            terms.remove('')
            
        preprocessed_terms=[]    # preprocess them
        for term in terms:
            preprocessed_terms.append(nlp_techniques.preprocess(term))

        # store the codes, terms, and preprocessed terms
        results=[results[0], terms, preprocessed_terms]              # results = [[codes], [all terms], [preprocessed terms]]

        m2_proposal_skills_mapper[proposal][skill]=results
        
# End time
print("End time:\t", datetime.datetime.now())

Start time:	 2026-04-01 15:28:41.787890
home_dir ./mapper4_main/data/
['G.1.3'] [['Numerical Linear Algebra', ' Conditioning', ' Determinants**', ' Eigenvalues and eigenvectors (direct and iterative methods) (REVISED)', ' Error analysis', ' Linear systems (direct and iterative methods)', ' Matrix inversion', ' Pseudoinverses**', ' Singular value decomposition (NEW)', ' Sparse, structured, and very large systems (direct and iterative methods) (REVISED)', '']]
home_dir ./mapper4_main/data/
['C.2.5'] [['Local and Wide-Area Networks (REVISED)', ' Access schemes', ' Buses', '']]
home_dir ./mapper4_main/data/
['H.2.8'] [['Database Applications', ' Data mining (NEW)', ' Image databases (NEW)', ' Scientific databases (NEW)', ' Spatial databases and GIS (NEW)', ' Statistical databases (NEW)', '']]
home_dir ./mapper4_main/data/
['J.7'] [['COMPUTERS IN OTHER SYSTEMS (C.3)', ' Command and control', ' Consumer products', ' Industrial control', ' Military', ' Process control', ' Publishing', ' Real 

In [14]:
# m2_proposal_skills_mapper['https://www.nsf.gov/pubs/2021/nsf21598/nsf21598.htm']

In [15]:
# export m2_proposal_mapper_skills
csv_m2_proposal_skills_mapper=[]
for i in m2_proposal_skills_mapper:   # proposal
    for j in m2_proposal_skills_mapper[i]: # skill
        csv_m2_proposal_skills_mapper.append([i, j, m2_proposal_skills_mapper[i][j][0], m2_proposal_skills_mapper[i][j][1], m2_proposal_skills_mapper[i][j][2]])   # [proposal, skill, code1, terms1]

csv_m2_proposal_skills_mapper=pd.DataFrame(csv_m2_proposal_skills_mapper, columns = ['nsf_proposal_links_v1', 'skill', 'code', 'terms', 'preprocessed_terms'])
csv_m2_proposal_skills_mapper.to_csv(save_dir+'m2_proposal_skills_mapper.csv', encoding='utf-8')
del csv_m2_proposal_skills_mapper

### Step 4.2 - For each of the researchers' skills, determine their appropriate ACM/JEL classifications. <a id='method_import_4.2'></a> 

<ul>
    <li>For each of the researchers' skills, identify what category they belong to.</li>
</ul>

In [ ]:
# Start time
print("Start time:\t", datetime.datetime.now())

# for each of the skills in researchers, identify what category they belong to.
m2_researcher_skills_mapper={}   # {researcher: [code, term]}

# for each researcher
for researcher in m2_researcher_skills:
    
    m2_researcher_skills_mapper[researcher]={}
    # for each skill
    for skill in m2_researcher_skills[researcher]:
        
        # call mapper
        results=list(M2.callMapper(skill, 0.3, "acm"))    # results = [[codes], [[terms1 for code1], [...], ...] => too complicated 
        
        # make the terms more understandable
        terms=[i for sublist in results[1] for i in sublist]
        while '' in terms:
            terms.remove('')
            
        preprocessed_terms=[]    # preprocess them
        for term in terms:
            preprocessed_terms.append(nlp_techniques.preprocess(term))

        # store the codes, terms, and preprocessed terms
        results=[results[0], terms, preprocessed_terms]      # results = [[codes], [all terms], [preprocessed terms]]
        
        m2_researcher_skills_mapper[researcher][skill]=results
        
# End time
print("End time:\t", datetime.datetime.now())

Start time:	 2026-04-01 15:31:17.160914
home_dir ./mapper4_main/data/
['E.3'] [['DATA ENCRYPTION', ' Code breaking (NEW)', ' Data encryption standard (DES)**', ' Public key cryptosystems', ' Standards (e.g., DES, PGP, RSA) (NEW)', '']]
home_dir ./mapper4_main/data/
['F.2.2'] [['Nonnumerical Algorithms and Problems (E.2-5, G.2, H.2-3)', ' Complexity of proof procedures', ' Computations on discrete structures', ' Geometrical problems and computations', ' Pattern matching', ' Routing and layout', ' Sequencing and scheduling', ' Sorting and searching', '']]
home_dir ./mapper4_main/data/
['A.0'] [['GENERAL', ' Biographies/autobiographies', ' Conference proceedings', ' General literary works (e.g., fiction, plays)', '']]
home_dir ./mapper4_main/data/
['E.1'] [['DATA STRUCTURES', ' Arrays', ' Distributed data structures (NEW)', ' Graphs and networks (REVISED)', ' Lists, stacks, and queues (REVISED)', ' Records (NEW)', ' Tables**', ' Trees', '']]
home_dir ./mapper4_main/data/
['I.2'] [['ARTIFI

In [ ]:
m2_researcher_skills_mapper['Agostinelli, Forest']

In [ ]:
# export m2_researcher_mapper_skills
csv_m2_researcher_skills_mapper=[]
for i in m2_researcher_skills_mapper:   # researcher
    for j in m2_researcher_skills_mapper[i]: # skill
        csv_m2_researcher_skills_mapper.append([i, j, m2_researcher_skills_mapper[i][j][0], m2_researcher_skills_mapper[i][j][1], m2_researcher_skills_mapper[i][j][2]])   # [researcher, skill, code1, terms1, preprocessedterms1]

csv_m2_researcher_skills_mapper=pd.DataFrame(csv_m2_researcher_skills_mapper, columns = ['researcher', 'skill', 'code', 'terms', 'preprocessed_terms'])
csv_m2_researcher_skills_mapper.to_csv(save_dir+'m2_researcher_skills_mapper.csv', encoding='utf-8')
del csv_m2_researcher_skills_mapper

### Step 4.3 - Check what categories each of the proposal/researchers' skills belong to. <a id='method_import_4.3'></a> 

In [ ]:
# check what categories (e.g., "I.2.x" - cut off the last part) each of the skills belong to, for proposals

m2_proposal_mapper_categories={}

for proposal in m2_proposal_skills_mapper:  # proposal
    m2_proposal_mapper_categories[proposal]=[]     
    for skill in m2_proposal_skills_mapper[proposal]:    # skill
        category=m2_proposal_skills_mapper[proposal][skill][0]
        
        # the loop is just in case a category has multiple categories in it (e.g., "['J.4', 'K.6.0']")
        for i in range(len(category)):
            if len(category[i])>3:
                category[i]=category[i][:3]
            m2_proposal_mapper_categories[proposal].append(category[i])  # save
            
# example
# m2_proposal_mapper_categories['https://www.nsf.gov/pubs/2021/nsf21598/nsf21598.htm']

In [ ]:
# check what categories (e.g., "I.2.x" - cut off the last part) each of the skills belong to, for researchers

m2_researcher_mapper_categories={}

for researcher in m2_researcher_skills_mapper:  # researcher
    m2_researcher_mapper_categories[researcher]=[]     
    for skill in m2_researcher_skills_mapper[researcher]:    # skill
        category=m2_researcher_skills_mapper[researcher][skill][0]
        
        # the loop is just in case a category has multiple categories in it (e.g., "['J.4', 'K.6.0']")
        for i in range(len(category)):
            if len(category[i])>3:
                category[i]=category[i][:3]
            m2_researcher_mapper_categories[researcher].append(category[i])  # save
            
# example
m2_researcher_mapper_categories['Agostinelli, Forest']

In [ ]:
# export m2_proposal_mapper_categories
csv_m2_proposal_mapper_categories=[]
for i in m2_proposal_mapper_categories:
    csv_m2_proposal_mapper_categories.append([i, m2_proposal_mapper_categories[i]])

csv_m2_proposal_mapper_categories=pd.DataFrame(csv_m2_proposal_mapper_categories, columns = ['proposal', 'skill_categories'])
csv_m2_proposal_mapper_categories.to_csv(save_dir+'m2_proposal_mapper_categories.csv', encoding='utf-8')
del csv_m2_proposal_mapper_categories

# export m2_researcher_mapper_categories
csv_m2_researcher_mapper_categories=[]
for i in m2_researcher_mapper_categories:
    csv_m2_researcher_mapper_categories.append([i, m2_researcher_mapper_categories[i]])

csv_m2_researcher_mapper_categories=pd.DataFrame(csv_m2_researcher_mapper_categories, columns = ['researcher', 'skill_categories'])
csv_m2_researcher_mapper_categories.to_csv(save_dir+'m2_researcher_mapper_categories.csv', encoding='utf-8')
del csv_m2_researcher_mapper_categories

***
### Step 5 - Create teams for each proposal call and for each researcher based on proposal/researcher mapper matches <a id='method_import_5'></a>

<ul>
    <li><a href="#method_import_5.1">5.1.</a> For each proposal, identify candidate researchers.</li>
    <li><a href="#method_import_5.2">5.2.</a> Taking the list from Step 5.1, form teams (pick any k out of a total N researchers).</li>
</ul>

### Step 5.1 - For each proposal, identify candidate researchers. <a id='method_import_5.1'></a>

In [ ]:
# Start time
print("Start time:\t", datetime.datetime.now())

m2_proposal_researcher_candidates={}  # {proposal: [researcher1, researcher2, ...]} - candidate researchers, not a fixed team

# for proposal in proposal-mapper list
for proposal in m2_proposal_mapper_categories:
    
    m2_proposal_researcher_candidates[proposal]=[]
    
    # for researcher in proposal-researcher list
    for researcher in m2_researcher_mapper_categories:
        flag=False
        
        # for each researcher skill
        for skill in m2_researcher_mapper_categories[researcher]:
            
            # check if skill in proposal-mapper (demand) skills
            if skill in m2_proposal_mapper_categories[proposal]:
                m2_proposal_researcher_candidates[proposal].append(researcher)
                break

                    
# End time
print("End time:\t", datetime.datetime.now())

### Step 5.2 - Taking the list from <a href='#method_import_5.1'>Step 5.1</a>, form teams (pick any k out of a total N researchers). <a id='method_import_5.2'></a>

In [ ]:
importlib.reload(M2)

# Start time
print("Start time:\t", datetime.datetime.now())

m2_teaming={}

# for each proposal
for proposal in m2_proposal_mapper_categories:
    
    m2_teaming[proposal]={}
    
    #for each target researcher
    for target_researcher in m2_researcher_mapper_categories:
        
        # create teams
        candidate_researchers=m2_proposal_researcher_candidates[proposal]
        num_of_teams=10
        teams=M2.create_teams_for_each_person(candidate_researchers,target_researcher,num_of_teams)
        
        # save
        m2_teaming[proposal][target_researcher]=teams
        
# End time
print("End time:\t", datetime.datetime.now())

In [ ]:
importlib.reload(M2)
# M2.create_teams_for_each_person(candidate_researchers,target_researcher,num_of_teams)

In [ ]:
# m2_teaming['https://www.nsf.gov/pubs/2021/nsf21598/nsf21598.htm']['Agostinelli, Forest']

### Step 6 - Apply Ultra-Metric. <a id='method_import_6'></a> 

In [ ]:
# Start time
print("Start time:\t", datetime.datetime.now())

# import Ultra-Metric 
import metrics_scorer as metrics

m2_goodness_scores={}

# for each proposal
for proposal in m2_teaming:
    #initialize
    m2_goodness_scores[proposal]={}
    
    # for each researcher
    for researcher in m2_teaming[proposal]:
        m2_goodness_scores[proposal][researcher]=[]
        goodness_for_each_researcher=[]
        
        # for each team
        for team in m2_teaming[proposal][researcher]: 
            
            # Apply Ultra-Metric (demand, team, researchers)
            temp_team_goodness=M2.apply_ultra_metric(m2_proposal_mapper_categories[proposal], team, m2_researcher_mapper_categories)
            
            # save to scores
            goodness_for_each_researcher.append(temp_team_goodness)
    
        # save to overall dictionary
        m2_goodness_scores[proposal][researcher]=goodness_for_each_researcher

# End time
print("End time:\t", datetime.datetime.now())

In [ ]:
m2_goodness_scores[proposal]['Agostinelli, Forest']

In [ ]:
# export m2_goodness_scores
csv_m2_goodness_scores=[]
for i in m2_goodness_scores:   # proposal
    for j in m2_goodness_scores[i]:   # researcher
        for k in m2_goodness_scores[i][j]:
            csv_m2_goodness_scores.append([i, j, k])
            
csv_m2_goodness_scores=pd.DataFrame(csv_m2_goodness_scores, columns = ['nsf_proposal_links_v1', 'researcher_name', 'goodness'])
csv_m2_goodness_scores.to_csv(save_dir+'m2_goodness_scores.csv', encoding='utf-8')
del csv_m2_goodness_scores

### Step 7 - Final exports to CSV. <a id='method_import_7'></a> 

<b>Variables:</b>
<ol>
    <li>[DONE] <i>m1_teaming{}</i> - {proposal_link: [[researcher1, [team1, team2, ...]], [researcher2, [team1, team2,...]]]</li>
    <li>[DONE] <i>m1_proposal_skills{}</i> - {proposal_link: [skill1, skill2, ...]}</li>
    <li>[DONE] <i>m1_researcher_skills{}</i> - {researcher: [skill1, skill2, ...]}</li>
    <li>[DONE] <i>m1_goodness{}</i> - {proposal_link: [[researcher1, [goodness1, ...], [researcher2, [goodness1, ...]]</li>
    <li>Complete teaming data </li>
</ol>

In [ ]:
# data = [proposal_link, proposal_title, proposal_skills, researcher, team, goodness]

# Start time
print("Start time:\t", datetime.datetime.now())

# group together and export the teaming data
save_dir="../data/v1_output_teaming/teaming_1698proposals_316researchers/"
csv_uc1_m2_teaming=[]
for i in m2_teaming:    # proposal
    for j in m2_teaming[i]:     # researcher
            
        # get title
        title_index=list(proposal_info['nsf_proposal_links_v1']).index(i)
        title=proposal_info['title'][title_index]

        # formatting variables (proposal year, proposal ID, proposal name + year)
        year=i.split("/")[4]
        proposal_id=i.split("/")[5]      # nsf#####
        csv_hyperlink_text=proposal_info['title'][title_index]+" ("+str(year)+")"     # Sample Proposal Name (2023)

        # sort teams in descending order (based on goodness scores)
        unsorted_teams=m2_teaming[i][j]
        unsorted_goodness=m2_goodness_scores[i][j]

        sorted_teams=[x for _,x in sorted(zip(unsorted_goodness, unsorted_teams), reverse=True)]
        sorted_goodness=sorted(unsorted_goodness, reverse=True) 
        
        # round goodness scores
        rounded_scores=[]
        for score in sorted_goodness:
            rounded_scores.append(round(score,4))
        
        # save
        csv_uc1_m2_teaming.append([proposal_id,
                                   year,
                                   i,
                                   #"=HYPERLINK(\""+i+"\", \""+csv_hyperlink_text+"\")",
                                   csv_hyperlink_text,
                                   m2_proposal_mapper_categories[i], 
                                   j,
                                   sorted_teams,
                                   rounded_scores])

csv_uc1_m2_teaming=pd.DataFrame(csv_uc1_m2_teaming, columns = ['proposal_id', 'year', 'proposal_link', 'title', "skills", "researcher_name", "team", "goodness"])
csv_uc1_m2_teaming.to_csv(save_dir+'teaming_uc1_m2.csv', encoding='utf-8')


# End time
print("End time:\t", datetime.datetime.now())

In [ ]:
import pandas as pd
import numpy as np

# 1. Calculate 'Volume' (#T) per row
# 'team' is likely a list of teams. We count how many teams are in that list.
csv_uc1_m2_teaming['volume'] = csv_uc1_m2_teaming['team'].apply(lambda x: len(x))

# 2. Calculate 'Average Goodness' (G) per row
# 'goodness' is a list of scores. We take the mean of that list.
csv_uc1_m2_teaming['avg_goodness_per_row'] = csv_uc1_m2_teaming['goodness'].apply(lambda x: np.mean(x) if len(x) > 0 else 0)

# 3. Group by Researcher to get metrics per r_j
researcher_stats = csv_uc1_m2_teaming.groupby('researcher_name').agg({
    'avg_goodness_per_row': 'mean',
    'volume': 'mean'
}).reset_index()

# 4. Final Aggregation (The values for Table 3.6)
final_mean_g = researcher_stats['avg_goodness_per_row'].mean()
final_std_g = researcher_stats['avg_goodness_per_row'].std()
final_volume = researcher_stats['volume'].mean()

print(f"Average Goodness (G): {final_mean_g:.4f} ± {final_std_g:.4f}")
print(f"Average Volume (#T): {final_volume:.2f}")